In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Obter informações do backend com Qiskit

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou versões mais recentes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Esta página explica como usar o Qiskit para encontrar informações sobre seus backends disponíveis.
## Listar backends
Para visualizar os backends aos quais você tem acesso, você pode ver uma lista na [página de recursos de computação,](https://quantum.cloud.ibm.com/computers) ou usar o método [`QiskitRuntimeService.backends()`](../api/qiskit-ibm-runtime/qiskit-runtime-service#backends). Esse método retorna uma lista de instâncias [`IBMBackend`](../api/qiskit-ibm-runtime/ibm-backend):

> **Note:** - Se você estiver conectado a uma instância ou região específica, ou se você inicializou o serviço com uma instância ou região específica usando `QiskitRuntimeService()`, apenas os backends disponíveis para você nessa instância ou região serão retornados. Caso contrário, todos os backends disponíveis para você em qualquer instância e em qualquer região serão retornados.
> - A lista de backends retornada pode não ser a mesma exibida na página de [recursos de computação](https://quantum.cloud.ibm.com/computers) do IBM Quantum Platform. A lista na página de recursos de computação é sempre filtrada pela região selecionada no topo da página.
Para executar o código a seguir, certifique-se de que você já se autenticou no serviço. Consulte [Configure sua conta IBM Cloud](/guides/cloud-setup) para mais detalhes.

In [1]:
# Initialize your account
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_boston')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_miami')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_kingston')>]

The [`QiskitRuntimeService.backend()`](../api/qiskit-ibm-runtime/qiskit-runtime-service#backend) method (note that this is singular: *backend*) takes the name of the backend as the input parameter and returns an [`IBMBackend`](../api/qiskit-ibm-runtime/ibm-backend) instance representing that particular backend:

In [2]:
service.backend("ibm_fez")

<IBMBackend('ibm_fez')>

O método [`QiskitRuntimeService.backend()`](../api/qiskit-ibm-runtime/qiskit-runtime-service#backend) (observe que este é o singular: *backend*) recebe o nome do backend como parâmetro de entrada e retorna uma instância [`IBMBackend`](../api/qiskit-ibm-runtime/ibm-backend) que representa aquele backend específico:

In [3]:
# Optionally pass in an instance, region, or both, to
# further filter the backends.
service = QiskitRuntimeService()

service.backends(simulator=False, operational=True, min_num_qubits=5)

[<IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_boston')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_miami')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_kingston')>]

Use these keyword arguments to filter by any attribute in backend configuration ([JSON schema](https://github.com/Qiskit/ibm-quantum-schemas/blob/main/schemas/backend_configuration_schema.json)) or status ([JSON schema](https://github.com/Qiskit/ibm-quantum-schemas/blob/main/schemas/backend_status_schema.json)). A similar method is [`QiskitRuntimeService.least_busy()`](../api/qiskit-ibm-runtime/qiskit-runtime-service#least_busy), which takes the same filters as `backends()` but returns the backend that matches the filters and has the least number of jobs pending in the queue:

In [4]:
service.least_busy(operational=True, min_num_qubits=5)

<IBMBackend('ibm_torino')>

## Filtrar backends
Você também pode filtrar os backends disponíveis pelas suas propriedades. Para filtros mais gerais, defina o argumento `filters` como uma função que aceita um objeto de backend e retorna `True` se ele atender aos seus critérios. Consulte a [documentação da API](../api/qiskit-ibm-runtime/qiskit-runtime-service#backends) para mais detalhes.

O código a seguir retorna apenas os backends que atendem a estes critérios e estão disponíveis para você _na instância atualmente selecionada_:

*   São dispositivos quânticos reais (`simulator=False`)
*   Estão atualmente operacionais (`operational=True`)
*   Têm pelo menos 5 qubits (`min_num_qubits=5`)

In [5]:
backend = service.backend("ibm_fez")

print(
    f"Name: {backend.name}\n"
    f"Version: {backend.version}\n"
    f"No. of qubits: {backend.num_qubits}\n"
)

Name: ibm_fez
Version: 2
No. of qubits: 156



For a full list of attributes, see the [`IBMBackend` API documentation](/docs/api/qiskit-ibm-runtime/ibm-backend).

## Native gates and operations

Each [processor family](/docs/guides/processor-types) has a native gate set. By default, the QPUs in each family only support running the gates and operations in the native gate set. Thus, every gate in the circuit must be translated (by the transpiler) to the elements of this set.

You can view the native gates and operations for a QPU either [with Qiskit](#native-gates-with-qiskit), or on the IBM Quantum&reg; Platform [Compute resources page](/docs/guides/qpu-information#native-gates-on-platform).

<span id="native-gates-with-qiskit"></span>
```python

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

for backend in service.backends():
    config = backend.configuration()
    if "simulator" in config.backend_name:
        continue
    print(f"Backend: {config.backend_name}")
    print(f"    Processor type: {config.processor_type}")
    print(f"    Supported instructions:")
    for instruction in config.supported_instructions:
        print(f"        {instruction}")
    print()
```

## Dynamic backend information

Backends can also have properties that change whenever the backed is calibrated, such as qubit frequency and operation error rates. Backends are usually calibrated every 24 hours, and their properties update after the calibration sequence completes. These properties can be used when optimizing quantum circuits or to construct noise models for a classical simulator.


### Qubit properties


The `backend.properties().qubit_property()` returns information about the qubits' physical attributes. It contains a dictionary of various properties of the qubit, each paired with its value and the timestamp of the last calibration.

- `T1 (Relaxation Time)`: The T1 time represents the average duration a qubit remains in its excited state $|1\rangle$ before decaying to its ground state $|0\rangle$ due to energy relaxation. This parameter is used to characterize the qubit's energy relaxation behavior, and is expressed in units of seconds (s).

- `T2 (Dephasing Time)`: The T2 time denotes the timescale over which a qubit maintains phase coherence of a superposition between the $|0\rangle$ and $|1\rangle$ states. It accounts for both energy relaxation and pure dephasing processes, providing insight into the qubit's coherence properties.

- `frequency`: This parameter specifies the resonant frequency of the qubit, indicating the energy difference between the $|0\rangle$ and $|1\rangle$ states, expressed in hertz (Hz).

- `anharmonicity`: Anharmonicity is the difference in energy between the first and second excited states of the qubit, also expressed in hertz (Hz).

- `readout_error`: The readout error quantifies the average probability of incorrectly measuring a qubit's state. It is commonly calculated as the mean of prob_meas0_prep1 and prob_meas1_prep0, providing a single metric for measurement fidelity.

- `prob_meas0_prep1`: This parameter indicates the probability of measuring a qubit in the 0 state when it was intended to be prepared in the $|1\rangle$ state, denoted as $P(0 | 1)$. It reflects errors in state preparation and measurement (SPAM), particularly measurement errors in superconducting qubits.

- `prob_meas1_prep0`: Similarly, this parameter represents the probability of measuring a qubit in the 1 state when it was intended to be prepared in the $|0\rangle$ state, denoted as $P(1 | 0)$. Like prob_meas0_prep1, it reflects SPAM errors, with measurement errors being the predominant contributor in superconducting qubits.

- `readout_length`: The readout_length specifies the duration of the readout operation for a qubit. It measures the time from the initiation of the measurement pulse to the completion of signal digitization, after which the system is ready for the next operation. Understanding this parameter is crucial for optimizing circuit execution, especially when incorporating mid-circuit measurements.

In [6]:
# fundamental physical properties of qubit 1
backend.qubit_properties(1)

QubitProperties(t1=0.00023160183954439313, t2=0.0002759670226087048, frequency=None)

In [7]:
# calibration data with detailed properties of qubit 0
backend.properties().qubit_property(0)

{'T1': (5.199156952582205e-05,
  datetime.datetime(2026, 1, 14, 16, 18, 26, tzinfo=tzlocal())),
 'T2': (2.253552085985709e-05,
  datetime.datetime(2026, 1, 14, 16, 19, 6, tzinfo=tzlocal())),
 'readout_error': (0.013916015625,
  datetime.datetime(2026, 1, 14, 21, 38, 31, tzinfo=tzlocal())),
 'prob_meas0_prep1': (0.026123046875,
  datetime.datetime(2026, 1, 14, 21, 38, 31, tzinfo=tzlocal())),
 'prob_meas1_prep0': (0.001708984375,
  datetime.datetime(2026, 1, 14, 21, 38, 31, tzinfo=tzlocal())),
 'readout_length': (1.56e-06,
  datetime.datetime(2026, 1, 14, 21, 38, 31, tzinfo=tzlocal()))}

In [8]:
# Retrieve qubit properties
qubit_index = 126  # Replace with your qubit index
qubit_props = backend.properties().qubit_property(qubit_index)

# Access specific properties
t1 = qubit_props.get("T1", (None,))[0]
t2 = qubit_props.get("T2", (None,))[0]
frequency = qubit_props.get("frequency", (None,))[0]
anharmonicity = qubit_props.get("anharmonicity", (None,))[0]
readout_error = qubit_props.get("readout_error", (None,))[0]
prob_meas0_prep1 = qubit_props.get("prob_meas0_prep1", (None,))[0]
prob_meas1_prep0 = qubit_props.get("prob_meas1_prep0", (None,))[0]
readout_length = qubit_props.get("readout_length", (None,))[0]

print(f"Qubit {qubit_index} Properties:")
print(f"  T1: {t1} seconds")
print(f"  T2: {t2} seconds")
print(f"  Frequency: {frequency} Hz")
print(f"  Anharmonicity: {anharmonicity} Hz")
print(f"  Readout Error: {readout_error}")
print(f"  P(0 | 1): {prob_meas0_prep1}")
print(f"  P(1 | 0): {prob_meas1_prep0}")
print(f"  Readout Length: {readout_length} seconds")

Qubit 126 Properties:
  T1: 9.563335658857979e-05 seconds
  T2: 6.570556299807121e-05 seconds
  Frequency: None Hz
  Anharmonicity: None Hz
  Readout Error: 0.006591796875
  P(0 | 1): 0.009765625
  P(1 | 0): 0.00341796875
  Readout Length: 1.56e-06 seconds


## Informações estáticas do backend
Algumas informações sobre um backend não mudam com frequência, como o nome, a versão, o número de qubits e os tipos de recursos que ele suporta. Essas informações estão disponíveis como atributos do objeto `backend`.

A célula a seguir constrói uma descrição de um backend.

In [9]:
backend.target["cz"][(1, 0)]

InstructionProperties(duration=6.8e-08, error=0.007831625819164134)

The following cell shows the properties for a measurement operation (including the readout error) on qubit 0.

In [10]:
backend.target["measure"][(0,)]

InstructionProperties(duration=1.56e-06, error=0.013916015625)

Para uma lista completa de atributos, consulte a [documentação da API `IBMBackend`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/ibm-backend).
## Gates e operações nativas
Cada [família de processador](/guides/processor-types) possui um conjunto de gates nativo. Por padrão, os QPUs de cada família suportam apenas a execução dos gates e operações do conjunto nativo. Portanto, cada gate no Circuit deve ser traduzido (pelo Transpiler) para os elementos desse conjunto.

Você pode visualizar os gates e operações nativas de um QPU [com o Qiskit](#native-gates-with-qiskit) ou na página de [recursos de computação](/guides/qpu-information#native-gates-on-platform) do IBM Quantum&reg; Platform.

<span id="native-gates-with-qiskit"></span>